# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 15 kwietnia 2024

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu z poprzednich zajęć (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania analogicznie do tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia.


# 1.1 MiniZinc

In [ ]:
using Distributions

function make_dzn(n::Int, capacities::Vector{Int}, data_number::Int)
    
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    content = """
    ITEM = _(1..$n);
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """
    file = open("generated_data/knapsack_generated_$data_number.dzn", "w+")
    write(file, content)
    close(file)
end

In [ ]:
n_items = [5, 10, 20, 25, 30, 40, 50, 75, 100, 150]

for (i, n) in enumerate(n_items)
    capacities = rand(DiscreteUniform(100, 1000), 3)
    make_dzn(n, capacities, i)

end

In [ ]:
n_items = [5, 10, 20, 25, 30, 40, 50, 75, 100]


for (i, n) in enumerate(n_items)
    println("------------KNAPSACK PROBLEM $i------------")

    f = open("generated_data/knapsack_generated_$i.dzn", "r")
    for line in readlines(f)
        println(line)
    end
    close(f)

    g = open("knapsack_results/knapsack_$i.txt", "r")
    for line in readlines(g)
        println(line)
    end
    close(g)
    println("--------------------------------------------")

end


------------KNAPSACK PROBLEM 1------------
ITEM = _(1..5);
capacities = [985, 534, 988];
profits = [675, 262, 976, 36, 613];
weights = [20, 34, 65, 11, 26];
Items count = 5
knapsack_1 = {}
profit = 0
knapsack_2 = {}
profit = 0
knapsack_3 = {to_enum(ITEM,1), to_enum(ITEM,2), to_enum(ITEM,3), to_enum(ITEM,4), to_enum(ITEM,5)}
profit = 2562
--------------------------------------------
------------KNAPSACK PROBLEM 2------------
ITEM = _(1..10);
capacities = [919, 873, 624];
profits = [541, 542, 847, 338, 306, 729, 25, 368, 864, 461];
weights = [78, 84, 48, 71, 40, 90, 31, 45, 44, 32];
Items count = 10
knapsack_1 = {}
profit = 0
knapsack_2 = {}
profit = 0
knapsack_3 = {to_enum(ITEM,1), to_enum(ITEM,2), to_enum(ITEM,3), to_enum(ITEM,4), to_enum(ITEM,5), to_enum(ITEM,6), to_enum(ITEM,7), to_enum(ITEM,8), to_enum(ITEM,9), to_enum(ITEM,10)}
profit = 5021
--------------------------------------------
------------KNAPSACK PROBLEM 3------------
ITEM = _(1..20);
capacities = [661, 880, 553];
profits

# 1.2 Tabu Search

In [15]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Symbol,Int})
    if move[1] === :add
        return (:remove, move[2])
    else
        return (:add, move[2])
    end
end


function possible_moves(p::KnapsackProblem, x::Vector{Bool})
    move_list = Tuple{Symbol,Int}[]
    current_weight = sum(p.weights .* x)
    # add item
    for i in eachindex(x, p.weights)
        if !x[i] && current_weight + p.weights[i] <= p.capacity
            push!(move_list, (:add, i))
        end
    end
    # remove item
    for i in eachindex(x, p.weights)
        if x[i]
            push!(move_list, (:remove, i))
        end
    end
    return move_list
end



possible_moves (generic function with 1 method)

In [16]:

function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(3000, profits, weights)
end

kp1 = generate_problem()

KnapsackProblem(3000, [209, 781, 914, 297, 733, 600, 843, 852, 769, 409  …  219, 175, 575, 836, 226, 609, 516, 648, 669, 848], [85, 99, 68, 81, 49, 16, 36, 73, 37, 69  …  70, 38, 68, 40, 11, 23, 10, 26, 44, 91])

In [17]:
function test(kp)
    x0 = fill(false, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)
    println(findall(sol))
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end

test(kp1)

[2, 13, 15, 50, 64, 70, 77, 87]
Best objective: -719
Last iteration: 9


In [ ]:
struct MultiKnapsackProblem
    capacities::NTuple{3,Int}
    profits::Vector{Int}
    weights::Vector{Int}
end

function objective(p::MultiKnapsackProblem, x::Vector{Int})
    return -sum(p.profits[i] for i in eachindex(x) if x[i] != 0)
end

function is_feasible(p::MultiKnapsackProblem, x::Vector{Int})
    weights_used = (0, 0, 0)
    for i in eachindex(x)
        bag = x[i]
        if bag != 0
            weights_used = Base.setindex(weights_used, weights_used[bag] + p.weights[i], bag)
        end
    end
    all(weights_used[i] <= p.capacities[i] for i in 1:3)
end

function possible_moves(p::MultiKnapsackProblem, x::Vector{Int})
    moves = Tuple{Symbol,Int,Int}[]
    for i in eachindex(x)
        current = x[i]
        for new_bag in 0:3
            if new_bag != current
                # tymczasowa kopia, sprawdzamy czy będzie spełniać ograniczenia
                x_tmp = copy(x)
                x_tmp[i] = new_bag
                if is_feasible(p, x_tmp)
                    push!(moves, (:assign, i, new_bag))
                end
            end
        end
    end
    return moves
end

function apply!(x::Vector{Int}, move::Tuple{Symbol,Int,Int})
    _, idx, new_bag = move
    x[idx] = new_bag
    return x
end

function invert_move(p::MultiKnapsackProblem, move::Tuple{Symbol,Int,Int})
    return move 
end

function generate_multi_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    capacities = (3000, 3000, 3000)
    return MultiKnapsackProblem(capacities, profits, weights)
end




generate_multi_problem (generic function with 1 method)

In [21]:
p = generate_multi_problem()
x0 = fill(0, length(p.profits))  
s = TabuState(p, x0; buffer_length=30)
best = solve_tabu(p, s; iteration_limit=1000)
println("Best profit: ", -objective(p, best))


ArgumentError: ArgumentError: reducing over an empty collection is not allowed; consider supplying `init` to the reducer